# Poverty and Race in the MORPC 15-County Region

This notebook walks through a realistic analysis scenario using `morpc-census`:

1. **Variable discovery** — find the Census tables for poverty status by race
2. **Fetch** — retrieve data for all 15 MORPC counties
3. **Snapshot** — compare poverty rates by race/ethnicity group, 2023, with reliability filtering
4. **Time series** — track changes in non-white poverty rates, 2019–2023
5. **Map** — choropleth of non-white poverty change by county

> **ACS 5-year note:** Each vintage year covers a 5-year period (e.g., 2023 covers 2019–2023). Consecutive years overlap by four years and are not statistically independent measurements.

In [ ]:
from morpc.logs import config_logs

config_logs('morpc-poverty-race-demo.log', 'debug')

In [ ]:
from morpc_census import (
    Endpoint, Group, CensusAPI, DimensionTable,
    RACE_TABLE_MAP,
    fetch_geos_from_scope_sumlevel,
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

SUPPRESS_CV = 0.40

## 1. Variable discovery

> **Network required** — the cells below make live calls to the Census API.

Start with an `Endpoint` and search `endpoint.groups` for tables related to the topic.
The ACS 5-year survey has hundreds of groups; filtering by keyword narrows the field quickly.

In [ ]:
ep = Endpoint('acs/acs5', 2023)

{k: v['description'] for k, v in ep.groups.items() if 'poverty' in v['description'].lower()}

**B17001** — *Poverty Status by Sex by Age* — is the standard person-level poverty table.
Its variables are organized as a hierarchy: a universe total, then below-poverty and
at-or-above-poverty subtotals, each further split by sex and age bands.

In [ ]:
g = Group(ep, 'B17001')
print(f"Description : {g.description}")
print(f"Universe    : {g.universe}")
print()
estimate_vars = [
    (k, v['label'])
    for k, v in g.variables.items()
    if k.endswith('E') and not k.endswith('EA')
]
dict(estimate_vars[:6])

B17001 has racial iteration variants **B17001A–B17001I**, each with the same variable structure
but for a single race/ethnicity group.  `RACE_TABLE_MAP` maps the letter suffix to the race label.

`B17001H` (White Alone, Not Hispanic or Latino) is the standard reference group when computing
a "non-white" population by subtraction.

In [ ]:
for code, label in RACE_TABLE_MAP.items():
    print(f"  B17001{code} — {label}")

available = {f'B17001{c}': RACE_TABLE_MAP[c] for c in RACE_TABLE_MAP if f'B17001{c}' in ep.groups}
print(f"\n{len(available)} of {len(RACE_TABLE_MAP)} racial iteration groups available")
available

## 2. Fetching data

For the poverty-by-race snapshot we need two variables from each racial iteration table:

| Variable | Meaning |
|---|---|
| `B17001X_001E` | Total population for whom poverty status is determined |
| `B17001X_002E` | Population with income below the poverty level |

The Census API returns a **margin of error** (MOE) alongside every estimate.
MOEs are reported at the 90% confidence level and appear in the `moe` column of `.long`.

In [ ]:
b17001 = CensusAPI(ep, 'region15', group='B17001')
b17001.long[['geoidfq', 'name', 'variable_label', 'variable', 'estimate', 'moe']].head(6)

In [ ]:
race_vars = []
for code in RACE_TABLE_MAP:
    race_vars += [f'B17001{code}_001E', f'B17001{code}_002E']

race_pov = CensusAPI(ep, 'region15', variables=race_vars)
race_pov.long[['name', 'variable_label', 'variable', 'estimate', 'moe']].head(10)

## 3. Poverty by race — 2023 snapshot

For each racial group, the poverty rate is the share of people in that group whose income
falls below the federal poverty level:

$$\text{poverty rate} = \frac{\text{B17001X\_002 (below poverty)}}{\text{B17001X\_001 (total)}}$$

In [ ]:
wide_est = race_pov.long.pivot(
    index=['geoidfq', 'name'], columns='variable', values='estimate'
)
wide_moe = race_pov.long.pivot(
    index=['geoidfq', 'name'], columns='variable', values='moe'
)
wide_est.columns.name = None
wide_moe.columns.name = None

rates = pd.DataFrame(index=wide_est.index)
cvs  = pd.DataFrame(index=wide_est.index)

for code, label in RACE_TABLE_MAP.items():
    n     = wide_est[f'B17001{code}_001']
    x     = wide_est[f'B17001{code}_002']
    moe_n = wide_moe[f'B17001{code}_001']
    moe_x = wide_moe[f'B17001{code}_002']

    rate = x / n
    under_root = moe_x**2 - rate**2 * moe_n**2
    moe_rate = (
        under_root.where(under_root >= 0, moe_x**2 + rate**2 * moe_n**2).pow(0.5) / n
    )

    rates[label] = (rate * 100).round(1)
    cvs[label]   = (moe_rate / (1.645 * rate)).where(rate > 0).round(3)

rates = rates.reset_index(level='geoidfq', drop=True)
cvs   = cvs.reset_index(level='geoidfq', drop=True)
rates.index.name = 'County'
cvs.index.name   = 'County'
rates

### Reliability: coefficient of variation

The Census Bureau measures estimate reliability with the **coefficient of variation** (CV):

$$CV = \frac{MOE_{\text{rate}}}{1.645 \times \text{rate}}$$

where $MOE_{\text{rate}}$ is derived from the component MOEs using the Census Bureau's
ratio formula (falling back to the sum-in-quadrature form when the expression under the
radical is negative):

$$MOE_{\text{rate}} = \frac{1}{N} \sqrt{MOE_x^2 - \hat{p}^2 \cdot MOE_N^2}$$

A CV above **40 %** indicates the estimate is unreliable.
Such cells are replaced with `—` in the table below.

In [ ]:
suppressed = rates.where(cvs <= SUPPRESS_CV)

suppressed.style \
    .format('{:.1f}%', na_rep='—') \
    .background_gradient(cmap='YlOrRd', axis=None) \
    .set_caption(
        'Poverty Rate by Race/Ethnicity (%), MORPC Region — 2023 ACS 5-Year'
        '\n(— = suppressed: CV > 40%)'
    )

In [ ]:
whole_region = race_pov.long.groupby('variable')['estimate'].sum()

region_rates = {}
for code, label in RACE_TABLE_MAP.items():
    total = whole_region.get(f'B17001{code}_001', 0)
    below = whole_region.get(f'B17001{code}_002', 0)
    if total > 0:
        region_rates[label] = round(below / total * 100, 1)

region_series = pd.Series(region_rates).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
region_series.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Poverty Rate by Race/Ethnicity\nMORPC 15-County Region — 2023 ACS 5-Year')
ax.set_xlabel('Poverty Rate (%)')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(ax.patches, region_series):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 4. Non-white poverty over time (2019–2023)

To track change in poverty among non-white residents, subtract the White alone,
not Hispanic or Latino population (B17001H) from the all-persons total (B17001):

$$\text{non-white below poverty} = \text{B17001\_002} - \text{B17001H\_002}$$
$$\text{non-white total} = \text{B17001\_001} - \text{B17001H\_001}$$

Because the two components are independent survey estimates, their MOEs combine in quadrature:

$$MOE_{\text{diff}} = \sqrt{MOE_A^2 + MOE_B^2}$$

The derived rate MOE then uses the same ratio formula as in Section 3.

> **Network required** — the loop below fetches five ACS vintage years.

In [ ]:
ts_vars = ['B17001_001E', 'B17001_002E', 'B17001H_001E', 'B17001H_002E']

long_frames = []
for year in range(2019, 2024):
    ep_yr = Endpoint('acs/acs5', year)
    api   = CensusAPI(ep_yr, 'region15', variables=ts_vars)
    long_frames.append(api.long)

long_ts = pd.concat(long_frames, ignore_index=True)
long_ts[['name', 'reference_period', 'variable', 'estimate', 'moe']].head(8)

In [ ]:
est_ts = long_ts.pivot_table(
    index=['geoidfq', 'name', 'reference_period'],
    columns='variable', values='estimate', aggfunc='first',
).reset_index()
moe_ts = long_ts.pivot_table(
    index=['geoidfq', 'name', 'reference_period'],
    columns='variable', values='moe', aggfunc='first',
).reset_index()
est_ts.columns.name = None
moe_ts.columns.name = None

nw_below = est_ts['B17001_002'] - est_ts['B17001H_002']
nw_total = est_ts['B17001_001'] - est_ts['B17001H_001']
est_ts['nw_rate'] = (nw_below / nw_total * 100).round(1)

moe_nw_below = (moe_ts['B17001_002']**2 + moe_ts['B17001H_002']**2).pow(0.5)
moe_nw_total = (moe_ts['B17001_001']**2 + moe_ts['B17001H_001']**2).pow(0.5)

rate_ts     = nw_below / nw_total
under_root  = moe_nw_below**2 - rate_ts**2 * moe_nw_total**2
moe_rate_ts = (
    under_root.where(under_root >= 0, moe_nw_below**2 + rate_ts**2 * moe_nw_total**2)
    .pow(0.5) / nw_total
)
est_ts['nw_cv'] = (moe_rate_ts / (1.645 * rate_ts)).where(rate_ts > 0).round(3)

est_ts.pivot_table(
    index='reference_period', columns='name', values='nw_rate', aggfunc='first'
)

Estimates with CV > 40 % are suppressed before plotting.
Gaps in a county's line indicate years where the non-white sample was too small
to produce a reliable estimate.

In [ ]:
pivot_rate = est_ts.pivot_table(
    index='reference_period', columns='name', values='nw_rate', aggfunc='first'
)
pivot_cv = est_ts.pivot_table(
    index='reference_period', columns='name', values='nw_cv', aggfunc='first'
)
pivot_plot = pivot_rate.where(pivot_cv <= SUPPRESS_CV)

fig, ax = plt.subplots(figsize=(12, 6))
pivot_plot.plot(ax=ax, marker='o', linewidth=1.5)
ax.set_title(
    'Non-White Poverty Rate by County\n'
    'MORPC Region — 2019–2023 ACS 5-Year Estimates\n'
    '(gaps = suppressed: CV > 40%)'
)
ax.set_ylabel('Poverty Rate (%)')
ax.set_xlabel('Vintage Year')
ax.set_xticks(range(2019, 2024))
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(
    title='County',
    labels=[n.replace(', Ohio', '') for n in pivot_plot.columns],
    bbox_to_anchor=(1.01, 1),
    loc='upper left',
    fontsize=8,
)
plt.tight_layout()
plt.show()

## 5. Mapping the change

`fetch_geos_from_scope_sumlevel` returns a GeoDataFrame with county boundaries.
Merging it with the computed poverty change gives a ready-to-plot choropleth.

A diverging colormap (red = increase, blue = decrease) makes the direction of change
immediately readable.

In [ ]:
change = est_ts.pivot_table(
    index=['geoidfq', 'name'],
    columns='reference_period',
    values='nw_rate',
    aggfunc='first',
)
change.columns = [f'rate_{yr}' for yr in change.columns]
change['change_pp'] = (change['rate_2023'] - change['rate_2019']).round(1)
change = change.reset_index()

geos = fetch_geos_from_scope_sumlevel('region15')

In [ ]:
geos_joined = geos.merge(change, left_on='GEOIDFQ', right_on='geoidfq', how='left')

max_abs = geos_joined['change_pp'].abs().max()

fig, ax = plt.subplots(figsize=(10, 8))
geos_joined.plot(
    column='change_pp',
    ax=ax,
    cmap='RdBu_r',
    vmin=-max_abs,
    vmax=max_abs,
    legend=True,
    legend_kwds={
        'label': 'Change in non-white poverty rate (pp)',
        'orientation': 'horizontal',
        'shrink': 0.6,
        'pad': 0.02,
    },
    edgecolor='white',
    linewidth=0.5,
)

for _, row in geos_joined.iterrows():
    cx = row.geometry.centroid.x
    cy = row.geometry.centroid.y
    label = row['NAME'].replace(', Ohio', '').replace(' County', '')
    val   = row['change_pp']
    ax.annotate(
        f"{label}\n{val:+.1f}pp",
        (cx, cy),
        ha='center', va='center', fontsize=6.5, color='black',
    )

ax.set_title(
    'Change in Non-White Poverty Rate\nMORPC 15-County Region — 2019 to 2023 ACS 5-Year',
    fontsize=13,
)
ax.axis('off')
plt.tight_layout()
plt.show()